In [3]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import os

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver


In [4]:
load_dotenv()


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)


In [5]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [6]:
def chat_node(state: ChatState):

    response = llm.invoke(state["messages"])

    return {
        "messages": [response]
    }


In [7]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
print("Type 'exit' to quit.\n")

thread_id = '1' # thread means one interaction with the chatbot

while True:

    query = input("You: ")

    if query.lower() == "exit":
        break


    config = {
        'configurable': {
            'thread_id': thread_id
        }
    }

    state = workflow.invoke(
        {
            "messages": [HumanMessage(content=query)]
        },

        config=config
    )

    print(f"\nAI: {state['messages'][-1].content}\n")

Type 'exit' to quit.


AI: It's nice to meet you. Is there something I can help you with or would you like to chat?


AI: Nice to meet you, Imran! How's your day going so far? Is there something on your mind that you'd like to talk about, or is this just a casual hello?


AI: I am an artificial intelligence language model, which means I'm a computer program designed to simulate conversation, answer questions, and provide information on a wide range of topics.

My technology is based on a type of AI called natural language processing (NLP), which allows me to understand and respond to human language. I've been trained on a massive dataset of text from various sources, including books, articles, and conversations, which enables me to generate human-like responses.

Some of my capabilities include:

* Answering questions on various topics, from science and history to entertainment and culture
* Generating text, such as stories, poems, or dialogues
* Translating text from one language to a

In [9]:
workflow.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='So hi', additional_kwargs={}, response_metadata={}, id='7b3be76c-91c5-430e-894f-77b225910fc3'), AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 37, 'total_tokens': 60, 'completion_time': 0.044786654, 'completion_tokens_details': None, 'prompt_time': 0.001673493, 'prompt_tokens_details': None, 'queue_time': 0.043705206, 'total_time': 0.046460147}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f88d9-e754-7f13-9ca6-ffa8f1be32df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 23, 'total_tokens': 60}), HumanMessage(content='Im imran butt', additional_kwargs={}, response_metadata={}, id